# Speaker utterance probabilities at obs = 2/5

For each speaker condition (`inf`, `pers+`, `pers-`), this notebook plots the
empirical distribution of utterances chosen by participants when they observed
**2 effective patients out of 5**, alongside the level-1 RSA speaker model's
predicted probabilities at the same observation. The model's α is fit by
pooled maximum likelihood across all 30 trials × 109 participants of the
matched scenario (i.e., for the `inf` panel we fit the α of a `psi='inf'`
model to all `inf`-scenario data, etc.).

This is the same kind of figure as §5.3 of the simulation analysis notebook
(`models/simulations/simulation_experiments/n1m5_T60_obs5000_seed42/analyze.ipynb`),
but with **bars showing the model's predicted distribution** at a pooled-fitted α,
and a **diamond marker for each utterance's empirical proportion** in the data.

At observation = 2/5, only three of the eight utterances are literally true
(`some,effective`, `some,ineffective`, `most,ineffective`). The remaining
five appear at zero in both the empirical and model distributions.


## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# Repo-root bootstrap so we can import the vendored RSA library.
HERE = Path.cwd().resolve()
# notebook lives in data/cogsci_rsa_speaker_experiment_n5/
sys.path.insert(0, str(HERE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from rsa_optimal_exp_core import World, PragmaticSpeaker_obs

# World matches the experiment: n=1 batch, m=5 patients per trial.
world = World(n=1, m=5)
print("Utterance vocabulary:", world.utterances)
print("Observation vocabulary (frequency tuples):", world.observations)


## Load data and collect per-scenario trial lists

For each speaker condition, build a flat list of `(num_effective, utterance)`
tuples across all 109 participants × 10 rounds. The CSV stores trials in wide
form (`{cond}_r{round}_num_effective` etc.); we melt them down.

In [ ]:
DATA_PATH = "./speaker_n1_fitted_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} participants from {DATA_PATH}")

# Map data labels to RSA utterance strings
PRED_MAP = {"Effective": "successful", "Ineffective": "unsuccessful"}

def num_effective_to_obs(n_eff):
    """num_effective k -> one-hot frequency tuple of length m+1=6."""
    return tuple(1 if i == n_eff else 0 for i in range(6))

def format_utterance(predicate, quantifier):
    """('Effective', 'Most') -> 'most,successful'."""
    return f"{quantifier.lower()},{PRED_MAP[predicate]}"

def collect_trials(df, scenario):
    """Return a list of (num_effective:int, utterance:str) for the given scenario."""
    trials = []
    for r in range(1, 11):
        sub = df[[f"{scenario}_r{r}_num_effective",
                  f"{scenario}_r{r}_predicate",
                  f"{scenario}_r{r}_quantifier"]].dropna()
        for _, row in sub.iterrows():
            trials.append((
                int(row[f"{scenario}_r{r}_num_effective"]),
                format_utterance(row[f"{scenario}_r{r}_predicate"], row[f"{scenario}_r{r}_quantifier"]),
            ))
    return trials

trials_by_scenario = {sc: collect_trials(df, sc) for sc in ["inf", "persp", "persm"]}
for sc, t in trials_by_scenario.items():
    n_obs2 = sum(1 for n, _ in t if n == 2)
    print(f"  {sc}: {len(t)} total trials, {n_obs2} at obs=2/5")


## Fit α by pooled maximum likelihood

For each scenario, sweep a log-spaced α grid and find the α that maximizes
the joint log-likelihood of all observed `(num_effective, utterance)` pairs
under the matched speaker model. Speaker config follows the same convention
as the simulation pipeline: `omega='strat'`, `update_internal=False`,
`beta=1.0` for `psi='inf'` (theoretical pure-info convention), `beta=0.0`
for `psi='pers+'` and `psi='pers-'` (pure persuasion).

In [ ]:
ALPHA_GRID = np.logspace(np.log10(0.5), np.log10(50), 80)

# Map scenario -> (psi, beta) for the speaker model.
SCENARIO_CFG = {
    "inf":   {"psi": "inf",   "beta": 1.0},
    "persp": {"psi": "pers+", "beta": 0.0},
    "persm": {"psi": "pers-", "beta": 0.0},
}

def fit_alpha_pooled(world, trials, psi, beta, alpha_grid, update_internal=False):
    """Return (best_alpha, best_ll). Pooled MLE across all trials."""
    obs_vocab = list(world.observations)
    utt_vocab = list(world.utterances)
    obs_to_col = {o: i for i, o in enumerate(obs_vocab)}
    utt_to_row = {u: i for i, u in enumerate(utt_vocab)}

    # Pre-index all trials.
    obs_idx = np.array([obs_to_col[num_effective_to_obs(n)] for n, _ in trials])
    utt_idx = np.array([utt_to_row[u] for _, u in trials])

    best_alpha, best_ll = None, -np.inf
    for alpha in alpha_grid:
        speaker = PragmaticSpeaker_obs(
            world=world, omega="strat", psi=psi,
            update_internal=update_internal, alpha=alpha, beta=beta,
        )
        # log P(u | obs) table — reorder to match (utt_vocab, obs_vocab).
        log_table = (speaker.utterance_log_prob_obs
                     .reindex(index=utt_vocab, columns=obs_vocab)
                     .to_numpy(dtype=np.float64))
        ll = float(np.sum(log_table[utt_idx, obs_idx]))
        if ll > best_ll:
            best_ll = ll
            best_alpha = float(alpha)
    return best_alpha, best_ll

fitted = {}
for sc, cfg in SCENARIO_CFG.items():
    a, ll = fit_alpha_pooled(world, trials_by_scenario[sc],
                              cfg["psi"], cfg["beta"], ALPHA_GRID)
    fitted[sc] = {"alpha": a, "ll": ll}
    print(f"  {sc} (psi={cfg['psi']}, beta={cfg['beta']}): "
          f"best α = {a:.3f},  pooled LL = {ll:.2f}")


## Empirical and model distributions at obs = 2/5

In [ ]:
PAPER_UTT_ORDER = [
    "all,unsuccessful",  "no,successful",
    "most,unsuccessful", "some,successful",
    "some,unsuccessful", "most,successful",
    "no,unsuccessful",   "all,successful",
]
UTT_LABELS = {
    "all,unsuccessful":  "all, ineffective",
    "no,successful":     "no, effective",
    "most,unsuccessful": "most, ineffective",
    "some,successful":   "some, effective",
    "some,unsuccessful": "some, ineffective",
    "most,successful":   "most, effective",
    "no,unsuccessful":   "no, ineffective",
    "all,successful":    "all, effective",
}

OBS_TARGET = 2  # 2 effective out of 5

def empirical_at_obs(trials, target_n_eff, vocab):
    counts = {u: 0 for u in vocab}
    for n_eff, utt in trials:
        if n_eff == target_n_eff and utt in counts:
            counts[utt] += 1
    total = sum(counts.values())
    return {u: counts[u] / total if total > 0 else 0.0 for u in vocab}, total

def model_at_obs(world, alpha, psi, beta, target_n_eff, vocab, update_internal=False):
    speaker = PragmaticSpeaker_obs(
        world=world, omega="strat", psi=psi,
        update_internal=update_internal, alpha=alpha, beta=beta,
    )
    obs = num_effective_to_obs(target_n_eff)
    log_p = speaker.utterance_log_prob_obs.reindex(index=vocab)[obs]
    return {u: float(np.exp(log_p[u])) for u in vocab}

results = {}
for sc, cfg in SCENARIO_CFG.items():
    emp, n = empirical_at_obs(trials_by_scenario[sc], OBS_TARGET, PAPER_UTT_ORDER)
    mod = model_at_obs(world, fitted[sc]["alpha"], cfg["psi"], cfg["beta"],
                       OBS_TARGET, PAPER_UTT_ORDER)
    results[sc] = {
        "empirical": emp, "model": mod,
        "alpha": fitted[sc]["alpha"], "n_trials": n,
    }
    print(f"  {sc}: n={n} trials at obs=2/5,  fitted α={fitted[sc]['alpha']:.3f}")
    for u in PAPER_UTT_ORDER:
        if emp[u] > 0 or mod[u] > 0.01:
            print(f"    {u:20s}  empirical={emp[u]:.3f}  model={mod[u]:.3f}")


## Plot

Three panels in a row (`inf`, `pers+`, `pers-`). Horizontal bars = **model predictions** at the pooled-fitted α; diamond markers = **empirical proportions**. First-listed utterance at the top.


In [ ]:
PANEL_TITLES = {
    "inf":   "Informative speaker",
    "persp": "Persuade-up speaker",
    "persm": "Persuade-down speaker",
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, sharey=True)
y_pos = np.arange(len(PAPER_UTT_ORDER))
bar_color = mcolors.to_rgb("steelblue")
diamond_color = "#FF3B3B"  # bright red

for ax, sc in zip(axes, ["inf", "persp", "persm"]):
    res = results[sc]
    emp = np.array([res["empirical"][u] for u in PAPER_UTT_ORDER])
    mod = np.array([res["model"][u]     for u in PAPER_UTT_ORDER])

    ax.barh(y_pos, mod, color=bar_color, edgecolor="black", linewidth=0.5,
            label=f"model (α={res['alpha']:.2f})")
    ax.scatter(emp, y_pos, marker="D", s=42, color=diamond_color,
               edgecolor="black", linewidth=0.5, zorder=3,
               label=f"empirical (n={res['n_trials']})")

    ax.set_yticks(y_pos)
    ax.set_yticklabels([UTT_LABELS[u] for u in PAPER_UTT_ORDER], fontsize=9)
    ax.set_title(PANEL_TITLES[sc], fontsize=11)
    ax.set_xlabel(f"P(utterance | obs = {OBS_TARGET}/5)")
    ax.set_xlim(-0.02, 1.02)
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(axis="x", alpha=0.25)

# First subplot inverts the shared y-axis (top label = first listed).
axes[0].invert_yaxis()
fig.suptitle(f"Speaker utterance preferences at obs = {OBS_TARGET}/5  "
             f"(empirical vs model under pooled-fitted α)", fontsize=12)
plt.tight_layout()
plt.show()

# Optionally save to ./figs/
SAVE = False
if SAVE:
    Path("./figs").mkdir(exist_ok=True)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(f"./figs/fig_speaker_utt_obs2.{ext}", bbox_inches="tight", dpi=150)
    print("Saved ./figs/fig_speaker_utt_obs2.{svg,pdf,png}")
